# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing dataset entities by their `@id` as prescribed by the Croissant standard.

### Dataset Source
The dataset is described using a Croissant schema and is accessible via a URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant metadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let’s review the available record sets, fields, and their `@id`s. We will enumerate the `@id` values for each record set and its fields for precise references in downstream analyses.

In [ ]:
# Get all record sets and their @id values
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in this dataset.")
else:
    for rs in record_sets:
        print(f"\nRecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(no name)')}")
        if 'fields' in rs:
            print("  Fields:")
            for field in rs['fields']:
                print(f"    - @id: {field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(unknown)')}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references are by Croissant `@id`.

If there is more than one record set, this will load all; otherwise, loads the single available record set.

In [ ]:
# Gather the @id for each record set available
record_set_ids = [rs['@id'] for rs in (dataset.metadata.record_sets or [])]

dataframes = {}
if len(record_set_ids) == 0:
    print("No record sets to extract records from.")
else:
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded DataFrame for RecordSet @id: {rs_id}")
        print(df.columns.tolist())
    # Preview the first record set loaded
    preview_id = record_set_ids[0]
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps by dynamically choosing a numeric field and group field via their `@id`s. This includes filtering, normalization, and grouping.

In [ ]:
# Proceed only if any record sets exist
if not dataframes:
    print("No tabular data available to process.")
else:
    # Select the first available record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Automatically detect numeric fields from the DataFrame (if present)
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")

        # Example filter
        mean_value = df[numeric_field_id].mean()
        threshold = mean_value if pd.notnull(mean_value) else 0

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by the first non-numeric column
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No categorical/group fields found for grouping.")
    else:
        print("No numeric fields detected in the record set.")

## 5. Visualization
Visualize the distribution of a numeric field, or the relationship between fields, for the first record set and numeric field available in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot only if numeric field was detected in the previous EDA
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field was found in EDA, show grouped boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR^2 dataset using the `mlcroissant` library, referencing all major entities by their Croissant `@id`. You loaded record sets, examined field identifiers, extracted tabular data, and performed exploratory data analysis, including normalization and visualization. This approach ensures repeatability and robust, standards-compliant data science workflows for FAIR datasets.